# PicoRV32 + CIM SoC — PYNQ 端 (无 pyserial, 配合 PC 串口读取)

**运行位置**: PYNQ-Z2 Jupyter Notebook  
**假设**: pyserial **不可用** (PYNQ 默认环境可能没有 pyserial)

## 工作模式

PYNQ 和 PC **同时**运行各自的 notebook:

| 端 | Notebook | 职责 |
|----|----------|------|
| PYNQ (本文件) | `picorv32_mlp_pynq_no_pyserial.ipynb` | Golden model 推理 + 导出结果, 最后加载 bitstream |
| PC | `picorv32_mlp_pc_no_pyserial.ipynb` | 读取 UART, 解析推理结果, 对比 golden |

## 重要: 执行顺序

Pure-PL bitstream 写入 `/dev/xdevcfg` 会重配整个 PL，
可能影响 PS 的时钟/MIO/DDR，导致 Jupyter kernel 卡死或断连。因此:

```
Step 1-2: Golden 推理 + 导出结果  (纯 Python, 不碰 PL, 安全)
Step 3:   [可选] PS 版交叉验证    (用 checkpoint1, 有 PS block design, 安全)
Step 4:   确认 checklist
Step 5:   加载 PicoRV32 bitstream (最后执行! PL 重配可能影响 PS)
```

## 硬件连接

USB-TTL 适配器的 USB 端插入 **PC** (不是 PYNQ):

```
PYNQ-Z2 PMODA              USB-TTL                PC
  Pin 1 (Y18, TX) -------> RXD                     |
  Pin 5 (GND)     -------> GND    ----USB---->  /dev/ttyUSB0
```

## 上传文件清单

| 文件 | 说明 |
|------|------|
| `cim_rv32_soc.bit` | PicoRV32 版 bitstream (checkpoint2) |
| `small_mlp_data/` | 模型数据目录 (权重/偏置/测试图片/量化参数) |
| `cim_soc.bit` + `cim_soc.hwh` | (可选) PS 版 bitstream 用于交叉验证 |

## 1. Golden Model 推理 (small_mlp_data)

纯 Python 计算，不触碰 PL，安全运行。
加载 `small_mlp_data/` 中的权重、偏置、量化参数和测试图片，
在 ARM PS 上做 bit-accurate INT8 推理。

In [ ]:
import subprocess, os, time, glob, struct
import numpy as np

DATA_DIR = "small_mlp_data"
BIT_FILE = "cim_rv32_soc.bit"

assert os.path.isdir(DATA_DIR), (
    f"{DATA_DIR}/ not found! Please upload.\n"
    "Generated by prepare_picorv32_env.ipynb on host PC."
)

# ---- Helper functions ----
def read_hex_u32(path):
    with open(path) as f:
        return [int(l.strip(), 16) for l in f if l.strip()]

def read_hex_u8(path):
    with open(path) as f:
        return [int(l.strip(), 16) & 0xFF for l in f if l.strip()]

def unpack_weights(chunk_path, out_dim, in_dim):
    chunks = read_hex_u32(chunk_path)
    w = np.zeros((out_dim, in_dim), dtype=np.int8)
    idx = 0
    for ob in range((out_dim + 15) // 16):
        for ib in range((in_dim + 15) // 16):
            for ch in range(64):
                r, cg = ch // 4, ch % 4
                word = chunks[idx]; idx += 1
                for b in range(4):
                    oi = ob * 16 + r
                    ii = ib * 16 + cg * 4 + b
                    if oi < out_dim and ii < in_dim:
                        val = (word >> (b * 8)) & 0xFF
                        w[oi, ii] = np.int8(struct.unpack('b', bytes([val]))[0])
    return w

def unpack_bias(path):
    return np.array([np.int32(np.uint32(v)) for v in read_hex_u32(path)], dtype=np.int32)

def hw_mvm(x_u8, w_i8, b_i32, zp, mult, shift, relu):
    """Bit-accurate INT8 MVM matching CIM hardware."""
    x_eff = np.clip(x_u8.astype(np.int32) - zp, 0, 511)
    acc = w_i8.astype(np.int32) @ x_eff + b_i32
    if relu:
        acc = np.maximum(acc, 0)
    out = np.zeros(len(acc), dtype=np.int32)
    for i in range(len(acc)):
        scaled = (acc[i] * mult) >> shift
        out[i] = max(-128, min(127, scaled))
    return out.astype(np.int8)

# ---- Load model ----
qp = read_hex_u32(f"{DATA_DIR}/quant_params.hex")
zps = read_hex_u32(f"{DATA_DIR}/zero_points.hex")
fc1_mult, fc1_shift = qp[0], qp[1]
fc2_mult, fc2_shift = qp[2], qp[3]
hw_zp1 = np.int32(np.uint32(zps[0]))
hw_zp2 = np.int32(np.uint32(zps[1]))

w1 = unpack_weights(f"{DATA_DIR}/fc1_weight_tiles.hex", 16, 784)
b1 = unpack_bias(f"{DATA_DIR}/fc1_bias.hex")
w2 = unpack_weights(f"{DATA_DIR}/fc2_weight_tiles.hex", 10, 16)
b2 = unpack_bias(f"{DATA_DIR}/fc2_bias.hex")

print(f"Model loaded from {DATA_DIR}/")
print(f"  FC1: {w1.shape}, FC2: {w2.shape}")
print(f"  Quant: fc1=({fc1_mult},{fc1_shift}), fc2=({fc2_mult},{fc2_shift})")
print(f"  ZP: fc1={hw_zp1}, fc2={hw_zp2}")

In [ ]:
# ---- Run golden inference on all test images ----
img_dir = f"{DATA_DIR}/test_images"
import re
img_files = sorted(glob.glob(f"{img_dir}/img_*.hex"))
# Keep only img_XXXX.hex (exclude img_XXXX_fc2.hex, img_XXXX_label.txt etc.)
img_files = [f for f in img_files if re.match(r'.*/img_\d{4}\.hex$', f)]
n_images = len(img_files)
print(f"Test images: {n_images}")
print()

golden_results = []
for img_path in img_files:
    base = os.path.basename(img_path).replace(".hex", "")
    label_path = os.path.join(img_dir, f"{base}_label.txt")
    img_u8 = np.array(read_hex_u8(img_path), dtype=np.uint8)
    label = int(open(label_path).read().strip())

    # FC1: 784->16, ReLU
    fc1_out = hw_mvm(img_u8, w1, b1[:16], hw_zp1, fc1_mult, fc1_shift, relu=True)
    fc1_u8 = fc1_out.astype(np.uint8)
    # FC2: 16->10, no activation
    fc2_out = hw_mvm(fc1_u8, w2, b2[:10], hw_zp2, fc2_mult, fc2_shift, relu=False)

    pred = int(np.argmax(fc2_out))
    logits = fc2_out.tolist()
    golden_results.append({
        "idx": base, "label": label, "pred": pred,
        "logits": logits, "correct": pred == label
    })

g_correct = sum(1 for r in golden_results if r["correct"])
print(f"Golden Model Accuracy: {g_correct}/{n_images} ({100*g_correct/n_images:.0f}%)")
print()
print(f"{'Image':>12} {'Label':>6} {'Pred':>5} {'Result':>8} {'Logits'}")
print("-" * 65)
for r in golden_results:
    status = "OK" if r["correct"] else "WRONG"
    logits_str = str(r['logits'])
    print(f"{r['idx']:>12} {r['label']:6d} {r['pred']:5d} {status:>8} {logits_str}")

## 2. 导出 Golden Results

保存 golden 结果为文本文件，下载到 PC 供 `picorv32_mlp_pc_no_pyserial.ipynb` 对比。

**这一步必须在加载 bitstream 之前完成!**

In [ ]:
GOLDEN_OUT = "pynq_golden_results.txt"

with open(GOLDEN_OUT, "w") as f:
    f.write("# PYNQ Golden Model Results (bit-accurate INT8)\n")
    f.write(f"# Generated: {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"# Model: small_mlp (784->16->10)\n")
    f.write(f"# Quant: fc1=({fc1_mult},{fc1_shift}), fc2=({fc2_mult},{fc2_shift})\n")
    f.write(f"# ZP: fc1={hw_zp1}, fc2={hw_zp2}\n")
    f.write(f"# Format: idx label pred correct logits...\n")
    for i, r in enumerate(golden_results):
        logits_str = " ".join(str(x) for x in r["logits"])
        ok = 1 if r["correct"] else 0
        f.write(f"{i} {r['label']} {r['pred']} {ok} {logits_str}\n")

print(f"Saved: {GOLDEN_OUT}")
print(f"  {n_images} images, accuracy {g_correct}/{n_images}")
print()
print(">>> Download this file to PC for cross-verification <<<")
print(f"  PC notebook expects: pynq_golden_results.txt")
print()
print("内容预览:")
with open(GOLDEN_OUT) as f:
    for line in f:
        print(f"  {line.rstrip()}")

## 3. [可选] PS 版 CIM 硬件交叉验证

如果有 checkpoint1 的 `cim_soc.bit` + `cim_soc.hwh`，
可以用 PS (ARM + MMIO) 驱动同一个 CIM IP 跑全部测试图。

checkpoint1 有 PS block design，`pynq.Overlay()` 可以安全加载。

In [ ]:
PS_BIT = "cim_soc.bit"
PS_HWH = "cim_soc.hwh"

if not all(os.path.exists(f) for f in [PS_BIT, PS_HWH]):
    print(f"Skip PS cross-validation: need {PS_BIT} + {PS_HWH} (checkpoint1)")
    ps_results = None
else:
    from pynq import Overlay, MMIO

    ol = Overlay(PS_BIT)
    print("PS Overlay loaded (safe: has PS block design)")
    mmio = MMIO(0x40000000, 0x4000)

    # Quick connectivity test
    mmio.write(0x010, 784)
    assert mmio.read(0x010) == 784, "AXI R/W failed!"
    print("AXI connectivity OK")

    # ---- CIM driver functions (same as firmware.c) ----
    def soft_reset():
        mmio.write(0x000, 0x4)

    def configure(in_dim, out_dim, zp, mult, shift, relu):
        mmio.write(0x010, in_dim)
        mmio.write(0x014, out_dim)
        mmio.write(0x018, (in_dim + 15) // 16)
        mmio.write(0x01C, (out_dim + 15) // 16)
        mmio.write(0x020, mult)
        mmio.write(0x024, shift)
        mmio.write(0x028, int(zp) & 0xFFFFFFFF)
        mmio.write(0x02C, 1 if relu else 0)

    def load_weights(chunks):
        mmio.write(0x044, 0)
        mmio.write(0x04C, 0x02)
        for c in chunks:
            mmio.write(0x048, int(c) & 0xFFFFFFFF)
        mmio.write(0x04C, 0x00)

    def load_bias(bias, n):
        for i in range(n):
            mmio.write(0x2000 + 4 * i, int(bias[i]) & 0xFFFFFFFF)

    def load_input(data):
        padded = ((len(data) + 15) // 16) * 16
        for i in range(padded):
            mmio.write(0x1000 + 4 * i, int(data[i]) if i < len(data) else 0)

    def start_wait(t=5.0):
        mmio.write(0x000, 0x2)
        mmio.write(0x000, 0x1)
        t0 = time.time()
        while not (mmio.read(0x004) & 0x2):
            if time.time() - t0 > t:
                raise TimeoutError("CIM compute timeout!")
        return time.time() - t0

    def read_output(n):
        return [np.int8(mmio.read(0x100 + 4 * i) & 0xFF) for i in range(n)]

    def read_pred():
        return mmio.read(0x040)

    # ---- Load weight data as raw chunks ----
    fc1_wc = read_hex_u32(f"{DATA_DIR}/fc1_weight_tiles.hex")
    fc1_bias_raw = read_hex_u32(f"{DATA_DIR}/fc1_bias.hex")
    fc2_wc = read_hex_u32(f"{DATA_DIR}/fc2_weight_tiles.hex")
    fc2_bias_raw = read_hex_u32(f"{DATA_DIR}/fc2_bias.hex")

    # ---- Run PS inference on all images ----
    ps_results = []
    for img_path in img_files:
        base = os.path.basename(img_path).replace(".hex", "")
        label_path = os.path.join(img_dir, f"{base}_label.txt")
        img_u8 = np.array(read_hex_u8(img_path), dtype=np.uint8)
        label = int(open(label_path).read().strip())

        # FC1
        soft_reset()
        configure(784, 16, hw_zp1, fc1_mult, fc1_shift, relu=True)
        load_weights(fc1_wc)
        load_bias(fc1_bias_raw, 16)
        load_input(img_u8)
        start_wait()
        fc1_out = np.array(read_output(16), dtype=np.uint8)

        # FC2
        configure(16, 10, hw_zp2, fc2_mult, fc2_shift, relu=False)
        load_weights(fc2_wc)
        load_bias(fc2_bias_raw, 10)
        load_input(fc1_out)
        start_wait()
        logits = read_output(10)
        pred = int(read_pred())

        ps_results.append({
            "idx": base, "label": label, "pred": pred,
            "logits": logits, "correct": pred == label
        })
        status = "OK" if pred == label else "WRONG"
        print(f"  {base}: label={label}, pred={pred} {status}")

    ps_correct = sum(1 for r in ps_results if r["correct"])
    print(f"\nPS CIM: {ps_correct}/{n_images} correct")

    # Check Golden vs PS
    all_match = True
    for i in range(n_images):
        if golden_results[i]["pred"] != ps_results[i]["pred"]:
            all_match = False
            print(f"  MISMATCH image {i}: golden={golden_results[i]['pred']}, PS={ps_results[i]['pred']}")
    if all_match:
        print(">>> Golden vs PS: ALL bit-exact match! <<<")

## 4. 加载前 Checklist

确认所有数据已保存，PC 端已就绪。

In [ ]:
print("=" * 60)
print(" Pre-bitstream Summary")
print("=" * 60)
print()
print(f"[Golden Model]  {g_correct}/{n_images} correct")
print(f"[Golden File]   {GOLDEN_OUT} ({'EXISTS' if os.path.exists(GOLDEN_OUT) else 'MISSING!'})")

try:
    if ps_results is not None:
        ps_c = sum(1 for r in ps_results if r["correct"])
        print(f"[PS CIM HW]     {ps_c}/{n_images} correct")
except NameError:
    print("[PS CIM HW]     Not run (need checkpoint1 files)")

print(f"[Bitstream]     {BIT_FILE} ({'EXISTS' if os.path.exists(BIT_FILE) else 'MISSING!'})")
print()
print("=" * 60)
print(" Checklist before loading bitstream:")
print("=" * 60)
print("  [1] Golden results saved?         ", "YES" if os.path.exists(GOLDEN_OUT) else "NO!")
print("  [2] Downloaded golden to PC?       Check manually")
print("  [3] PC UART reader running?        Must be ready BEFORE loading!")
print("  [4] USB-TTL wired correctly?       PMODA Pin1(TX)->RXD, Pin5->GND")
print()
print("If all OK, run the next Cell to load bitstream.")
print("WARNING: Jupyter may become unresponsive after loading Pure-PL bitstream.")

## 5. 加载 PicoRV32 Bitstream (最后执行!)

**WARNING**: Pure-PL bitstream 没有 PS block design。写入 `/dev/xdevcfg` 后
PL 完全重配置，可能影响 PS 的时钟/DDR/Ethernet，导致 Jupyter kernel 断连。

这就是为什么这个 Cell 放在最后——所有需要保存的结果都已经写入文件了。

**操作顺序**:
1. 确认 PC 端 UART 读取已启动
2. 运行本 Cell
3. 如果 Jupyter 卡住: PicoRV32 仍然在跑，UART 数据仍然在发，PC 端可以正常收到
4. 如果需要恢复 Jupyter: 按 PYNQ 板上的 PROG 按钮 或 重启板子

In [ ]:
assert os.path.exists(BIT_FILE), f"{BIT_FILE} not found! Please upload."

print("=" * 60)
print(f"  Loading {BIT_FILE} ({os.path.getsize(BIT_FILE)/1024:.0f} KB)")
print("=" * 60)
print()
print(">>> Confirm PC-side UART reader is running! <<<")
print()
print("WARNING: Pure-PL bitstream may cause Jupyter to hang.")
print("  If that happens, the UART output is still sent to PC.")
print("  Golden results are already saved to:", GOLDEN_OUT)
print()

# Method 1: /dev/xdevcfg (Zynq-7000 standard)
try:
    subprocess.run(
        ["sudo", "bash", "-c", f"cat {BIT_FILE} > /dev/xdevcfg"],
        check=True, timeout=30
    )
    print("Bitstream loaded!")
except subprocess.TimeoutExpired:
    print("xdevcfg write timed out (30s). Board may need reset.")
    print("But PicoRV32 may already be running -- check PC for UART output.")
except Exception as e:
    print(f"Method 1 failed: {e}")
    # Method 2: fpgautil
    try:
        subprocess.run(["fpgautil", "-b", BIT_FILE], check=True, timeout=30)
        print("Bitstream loaded! (fpgautil)")
    except subprocess.TimeoutExpired:
        print("fpgautil timed out. Check PC for UART output.")
    except Exception as e2:
        print(f"Method 2 failed: {e2}")
        print("Manual: sudo bash -c 'cat cim_rv32_soc.bit > /dev/xdevcfg'")

print()
print("If you can see this, PL reconfiguration did not kill Jupyter!")
print("PicoRV32 is now executing firmware.")
print("  LED0 (R14): CIM done indicator")
print("  LED1 (P14): heartbeat (~1Hz blink = CPU running)")
print()
print("UART output is being sent to PMODA Pin 1 (Y18) --> PC")
print("To re-send: press BTN0 on PYNQ board")

## 6. [可选] 复位 PicoRV32

如果 PC 端漏掉了 UART 输出:
- **方式 A**: 按 PYNQ 上的 **BTN0** (物理复位按钮)
- **方式 B**: 重新加载 bitstream (下方 Cell, 只在 Jupyter 仍存活时可用)

In [ ]:
RELOAD = False  # Set to True to reload

if RELOAD:
    print(">>> Confirm PC-side UART reader is running! <<<")
    print("Reloading bitstream in 3 seconds...")
    time.sleep(3)
    try:
        subprocess.run(
            ["sudo", "bash", "-c", f"cat {BIT_FILE} > /dev/xdevcfg"],
            check=True, timeout=30
        )
        print("Bitstream reloaded! PicoRV32 is running.")
    except Exception as e:
        print(f"Reload failed: {e}")
        print("Press BTN0 on the board instead.")
else:
    print("Set RELOAD = True to reload bitstream.")
    print("Or just press BTN0 on the PYNQ board.")

## 7. [可选] 上传 PC 端结果进行交叉验证

如果 Jupyter 仍然存活，可以上传 PC 端保存的 `pc_uart_result.txt` 进行对比。

In [ ]:
PC_RESULT = "pc_uart_result.txt"

if os.path.exists(PC_RESULT):
    print("--- PC UART Result (uploaded) ---")
    pc = {}
    with open(PC_RESULT) as f:
        for line in f:
            if line.startswith("#"): continue
            parts = line.strip().split()
            if len(parts) >= 2:
                pc[parts[0]] = parts[1]

    if "predicted" in pc and "expected" in pc:
        hw_pred = int(pc["predicted"])
        hw_label = int(pc["expected"])
        print(f"  HW: pred={hw_pred}, label={hw_label}")

        matched = [r for r in golden_results if r["label"] == hw_label]
        if matched:
            g = matched[0]
            print(f"  Golden: pred={g['pred']}")
            if hw_pred == g["pred"]:
                print("  >>> PicoRV32 vs Golden: MATCH (bit-exact!) <<<")
            else:
                print("  >>> PicoRV32 vs Golden: MISMATCH <<<")
        else:
            print(f"  No golden record for label={hw_label}")
else:
    print(f"{PC_RESULT} not found.")
    print("Upload from PC after running the PC-side notebook.")

## 总结

本 notebook 在 **PYNQ** 上完成:

1. **先** 用 small_mlp_data 做 Golden Model 推理 (纯 Python, 安全)
2. **先** 导出 `pynq_golden_results.txt` 供 PC 对比
3. (可选) PS 版 CIM 交叉验证 (checkpoint1, 有 PS block design, 安全)
4. **最后** 加载 PicoRV32 bitstream (Pure-PL, 可能影响 PS)

```
安全区 (Jupyter 稳定)                   危险区
  |                                       |
  |-- Golden inference (pure Python)      |
  |-- Save golden_results.txt             |
  |-- [PS cross-validation]               |
  |                                       |
  +---- barrier: all data saved ----------+
                                          |
                              Load Pure-PL bitstream
                              (may hang Jupyter)
                                          |
                              PicoRV32 UART --> PC
```

即使 Jupyter 在加载 bitstream 后挂掉，golden 结果已保存，
PC 端仍可正常接收 UART 并完成对比。